# Evaluating AR-LSTM with Data Assimilation

This notebook walks you through evaluating a trained Autoregressive LSTM (AR-LSTM) model on local data, and then running Data Assimilation (DA) to see if it improves predictions.

### Prerequisites
You need a trained AR-LSTM model. If you haven't trained one yet, you can do so by running the following command in your terminal from the `flood-forecasting` root directory:

```bash
python googlehydrology/run.py train --config-file tutorial/configs/train-arlstm-local.yml
```
This will train a model on 5 basins for 5 epochs and save the weights in `tutorial/model-runs/`.


In [ ]:
import os
import sys
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np

# Add parent directory to path so we can import googlehydrology
sys.path.append(str(Path(os.getcwd()).parent))
from googlehydrology.utils.config import Config
from googlehydrology.evaluation.evaluate import start_evaluation


### Step 1: Find the latest run directory

We will automatically find the latest run directory matching our experiment name `arlstm-local-example`.
If you haven't run training yet, this cell will warn you.


In [ ]:
# Find the latest run directory
run_dirs = glob.glob('model-runs/arlstm-local-example_*')
if not run_dirs:
    # Try looking in the tutorial folder if running from there
    run_dirs = glob.glob('tutorial/model-runs/arlstm-local-example_*')

if not run_dirs:
    raise RuntimeError("No run directories found. Please run training first!")

latest_run_dir = max(run_dirs, key=os.path.getmtime)
run_dir_path = Path(latest_run_dir).resolve()
print(f"Using run directory: {run_dir_path}")


### Step 2: Run Evaluation WITHOUT Data Assimilation

We will load the config from the run directory and run the standard evaluation. This will generate predictions without any state updates during test time.
Results will be saved in the run directory as `test_results.zarr`.


In [ ]:
# Load base config
config_baseline = Config(run_dir_path / 'config.yml')

# Ensure we are in CPU mode for local testing if needed
# config_baseline.device = 'cpu'

# Run evaluation without DA
print("Running baseline evaluation...")
start_evaluation(
    cfg=config_baseline,
    run_dir=run_dir_path,
    period='test',
    data_assimilation=False
)
print("Baseline evaluation complete. Results saved to test_results.zarr")


### Step 3: Run Evaluation WITH Data Assimilation

Now we will configure and run Data Assimilation.
We will dynamically add the `assimilation_config` to our configuration in python.
This configures the DA loop to optimize the LSTM initial states (`h_n`, `c_n`) for each assimilation window (length 1 day, using 10 days of history).
Results will be saved in the run directory as `test_results_data_assimilation.zarr`.


In [ ]:
# Load config again to avoid modifying the baseline config object
config_da = Config(run_dir_path / 'config.yml')

# Define Data Assimilation configuration
da_config_dict = {
    'assimilation_config': {
        'assimilation_lead_time': 0,      # simulation mode
        'assimilation_window': 1,         # update states daily
        'history': 10,                    # use 10 days of past observations for update
        'learning_rate': 0.05,            # learning rate for optimization
        'assimilation_targets': ['h_n', 'c_n'], # optimize hidden and cell states
        'epochs': 10,                     # max optimization epochs per window
        'loss': 'MSE',                    # loss function to minimize
        'optimizer': 'Adam'
    }
}

# Update config with DA settings
config_da.update_config(da_config_dict)

# Run evaluation with DA
print("Running evaluation with Data Assimilation...")
start_evaluation(
    cfg=config_da,
    run_dir=run_dir_path,
    period='test',
    data_assimilation=True
)
print("Data Assimilation evaluation complete. Results saved to test_results_data_assimilation.zarr")


### Step 4: Load and Compare Results

We will load both datasets (with and without DA) and plot the hydrographs for a basin to see the difference.
We will also compute the NSE (Nash-Sutcliffe Efficiency) to quantitatively compare them.


In [ ]:
# Load results
ds_no_da = xr.open_zarr(run_dir_path / 'test_results.zarr')
ds_da = xr.open_zarr(run_dir_path / 'test_results_data_assimilation.zarr')

# Inspect available basins
basins = list(ds_no_da.basin.values)
print(f"Available basins: {basins}")

# Select a basin to plot (e.g. the first one)
selected_basin = basins[0]

# Extract streamflow predictions (y_hat) and observations (y)
# In simulation mode, lead_time is 1 (or 0, depending on config)
# Let's check dims
print("No DA Dims:", ds_no_da.dims)

# Plot comparison
plt.figure(figsize=(12, 6))
# Observations (should be same in both)
plt.plot(ds_no_da.date, ds_no_da['streamflow_obs'].sel(basin=selected_basin), label='Observations', color='black', alpha=0.5)
# Predictions without DA
plt.plot(ds_no_da.date, ds_no_da['streamflow_sim'].sel(basin=selected_basin), label='Predictions (No DA)', color='blue')
# Predictions with DA
plt.plot(ds_da.date, ds_da['streamflow_sim'].sel(basin=selected_basin), label='Predictions (With DA)', color='red', linestyle='--')

plt.title(f"Hydrograph Comparison for Basin: {selected_basin}")
plt.xlabel("Date")
plt.ylabel("Streamflow")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Calculate and compare NSE metrics
from googlehydrology.evaluation.metrics import calculate_metrics

# Calculate NSE for the selected basin
obs = ds_no_da['streamflow_obs'].sel(basin=selected_basin).squeeze()
sim_no_da = ds_no_da['streamflow_sim'].sel(basin=selected_basin).squeeze()
sim_da = ds_da['streamflow_sim'].sel(basin=selected_basin).squeeze()

# Calculate metrics
metrics_no_da = calculate_metrics(obs, sim_no_da, metrics=['NSE'])
metrics_da = calculate_metrics(obs, sim_da, metrics=['NSE'])

print(f"NSE for basin {selected_basin}:")
print(f"  Without DA: {metrics_no_da['NSE']:.4f}")
print(f"  With DA:    {metrics_da['NSE']:.4f}")

# Calculate average NSE over all basins
nse_no_da_list = []
nse_da_list = []

for b in basins:
    o = ds_no_da['streamflow_obs'].sel(basin=b).squeeze()
    s_no_da = ds_no_da['streamflow_sim'].sel(basin=b).squeeze()
    s_da = ds_da['streamflow_sim'].sel(basin=b).squeeze()
    
    # Skip if all obs are NaN
    if o.isnull().all():
        continue
        
    m_no_da = calculate_metrics(o, s_no_da, metrics=['NSE'])
    m_da = calculate_metrics(o, s_da, metrics=['NSE'])
    
    # calculate_metrics might return NaN if there are issues
    if not np.isnan(m_no_da['NSE']):
        nse_no_da_list.append(m_no_da['NSE'])
    if not np.isnan(m_da['NSE']):
        nse_da_list.append(m_da['NSE'])

print("\nAverage NSE over all valid basins:")
print(f"  Without DA: {np.mean(nse_no_da_list):.4f}")
print(f"  With DA:    {np.mean(nse_da_list):.4f}")
